## Read raw data into pandas

## Configuration

In [ ]:
# ---------------------------------------------------------------------------
# Change DATA_VARIANT to switch between full and small datasets.
#   ""       → raw_profiles.csv / parsed_experiences.csv / classified_jobs_gemini.csv
#   "_small" → raw_profiles_small.csv / parsed_experiences_small.csv / classified_jobs_gemini_small.csv
# ---------------------------------------------------------------------------
DATA_VARIANT = ""  # set to "_small" for the small dataset

RAW_INPUT          = f"../data/raw/raw_profiles{DATA_VARIANT}.csv"
PARSED_OUTPUT      = f"../data/processed/parsed_experiences{DATA_VARIANT}.csv"
CLASSIFIED_OUTPUT  = f"../data/processed/classified_jobs_gemini{DATA_VARIANT}.csv"
SENIORITY_OUTPUT   = f"../data/processed/classified_jobs_gemini_seniority{DATA_VARIANT}.csv"
DURATION_OUTPUT    = f"../data/processed/classified_jobs_gemini_duration{DATA_VARIANT}.csv"
PROFILE_OUTPUT     = f"../data/processed/classified_jobs_gemini_profiles{DATA_VARIANT}.csv"

# ---------------------------------------------------------------------------
# Set to False to skip a step (e.g. if its output file already exists).
# ---------------------------------------------------------------------------
RUN_PARSING        = False   # parse raw profiles → parsed_experiences
RUN_CLASSIFICATION = False   # classify job titles → classified_jobs_gemini
RUN_SENIORITY      = False   # assign seniority levels → classified_jobs_gemini_seniority
RUN_DURATION       = False   # calculate role durations → classified_jobs_gemini_duration
RUN_PROFILES       = False   # classify profiles → classified_jobs_gemini_profiles

print(f"Using dataset variant: '{DATA_VARIANT or 'full'}'")
print(f"  Raw input:          {RAW_INPUT}")
print(f"  Parsed output:      {PARSED_OUTPUT}")
print(f"  Classified output:  {CLASSIFIED_OUTPUT}")
print(f"  Seniority output:   {SENIORITY_OUTPUT}")
print(f"  Duration output:    {DURATION_OUTPUT}")
print(f"  Profile output:     {PROFILE_OUTPUT}")
print()
print(f"Steps to run:")
print(f"  Parsing:        {'ON' if RUN_PARSING else 'SKIP'}")
print(f"  Classification: {'ON' if RUN_CLASSIFICATION else 'SKIP'}")
print(f"  Seniority:      {'ON' if RUN_SENIORITY else 'SKIP'}")
print(f"  Duration:       {'ON' if RUN_DURATION else 'SKIP'}")
print(f"  Profiles:       {'ON' if RUN_PROFILES else 'SKIP'}")

In [ ]:
import pandas as pd

raw = pd.read_csv(RAW_INPUT)
raw.head()

## Define the experience data class
The experience data class will correspond to one row in the resulting csv.

In [ ]:
from dataclasses import dataclass
from typing import Optional

@dataclass
class Experience:
    """Represents a single work experience entry from a LinkedIn profile.
    
    Designed to map directly to a CSV row.
    """
    profile_id: int
    job_title: str
    country: Optional[str]
    start_date: Optional[str]  # e.g. "2020-01" or "Jan 2020"
    end_date: Optional[str]    # None means present/current role
    company: Optional[str] = None  # Optional field for company name
    position_in_career: Optional[int] = None  # Optional field for position in career timeline

## Format the Experiences
Format the existing experiences in the format that matches the experience data class.

In [ ]:
import os
import json
import time
import re

from google import genai
from google.genai.errors import ClientError
from dotenv import load_dotenv

load_dotenv()


# ---------------------------------------------------------------------------
# GeminiClient – calls the Google Gemini API directly
# ---------------------------------------------------------------------------

class GeminiClient:
    """Thin wrapper around google-genai that exposes generate_text(prompt) -> str."""

    def __init__(self, model_name: str, api_key: str):
        self.model_name = model_name
        self._client = genai.Client(api_key=api_key)

    def generate_text(self, prompt: str) -> str:
        response = self._client.models.generate_content(
            model=self.model_name, contents=prompt
        )
        return response.text


# ---------------------------------------------------------------------------
# Setup
# ---------------------------------------------------------------------------

api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY", "")
MODEL = "gemini-2.5-flash-lite"
client = GeminiClient(model_name=MODEL, api_key=api_key)

PARSE_PROMPT = """You are a structured data extractor. Given raw LinkedIn experience text for one person,
extract each distinct job/position into a JSON array. The text is messy — it may contain duplicated
fragments, merged lines, location strings, employment types (Full-time, Part-time, etc.), and
date strings that appear twice.

For each job, extract:
- "job_title": the cleaned job title (e.g. "Software Engineer", "IT Manager")
- "company": the company/organisation name, or null if unclear
- "start_date": in "YYYY-MM" format if month is available, or "YYYY" if only year, or null
- "end_date": in "YYYY-MM" format, "YYYY", or null if marked "Present" or still current
- "is_current": true if this is a current/present position, false otherwise

Rules:
- If the job title is in any other language than English, translate it to English if possible. If translation is uncertain, keep the original.
- Ignore location strings, employment type labels (Full-time, Part-time, Hybrid, Remote, On-site),
  logo mentions, and duration strings (e.g. "6 yrs 3 mos").
- If the same position appears to be duplicated (same title, same company, same dates), output it only once.
- If a company umbrella header appears (e.g. "ING Hubs Romania Full-time · 6 yrs 9 mos"),
  treat it as context for the sub-positions below it, not as a separate job.
- Return ONLY the JSON array, no markdown fences, no explanation.

Raw experience text:
{experience_text}

JSON array:"""

MAX_RETRIES = 3
_RATE_LIMIT_DELAY = 60 / 13  # ≈4.6 s — stays under 15 req/min free-tier limit


def parse_experience_with_llm(profile_id: int, country: str, experience_text: str) -> list[Experience]:
    """Send raw experience text to the LLM and parse the JSON response into Experience objects.
    
    position_in_career is assigned so that 1 = oldest job, N = most recent job.
    LinkedIn lists jobs most-recent-first, so the first item in the array gets the highest index.
    """
    if pd.isna(experience_text) or not experience_text.strip():
        return []

    prompt = PARSE_PROMPT.format(experience_text=experience_text)

    for attempt in range(MAX_RETRIES):
        try:
            raw_text = client.generate_text(prompt).strip()
            # Strip markdown fences if present
            raw_text = re.sub(r"^```(?:json)?\s*", "", raw_text)
            raw_text = re.sub(r"\s*```$", "", raw_text)

            jobs = json.loads(raw_text)
            n = len(jobs)
            experiences = []
            for idx, job in enumerate(jobs):
                experiences.append(Experience(
                    profile_id=profile_id,
                    job_title=job.get("job_title", "Unknown"),
                    country=country,
                    start_date=job.get("start_date"),
                    end_date=job.get("end_date") if not job.get("is_current") else None,
                    company=job.get("company"),
                    position_in_career=n - idx,  # most recent (first in list) → highest index
                ))
            time.sleep(_RATE_LIMIT_DELAY)
            return experiences

        except (json.JSONDecodeError, KeyError) as e:
            print(f"  JSON parse error for profile {profile_id} (attempt {attempt+1}): {e}")
            if attempt < MAX_RETRIES - 1:
                time.sleep(5)
        except ClientError as e:
            if e.code == 429:
                wait = 60
                print(f"  Rate limited. Waiting {wait}s (attempt {attempt+1})...")
                time.sleep(wait)
            else:
                print(f"  API error for profile {profile_id}: {e}")
                break
        except Exception as e:
            print(f"  Unexpected error for profile {profile_id}: {e}")
            break

    # Fallback: return a single unknown entry so the profile isn't lost
    return [Experience(
        profile_id=profile_id,
        job_title="PARSE_ERROR",
        country=country,
        start_date=None,
        end_date=None,
        company=None,
        position_in_career=None,
    )]


if RUN_PARSING:
    # --- Run the LLM parser across all profiles ---
    all_experiences: list[Experience] = []

    for i, row in raw.iterrows():
        pid = row["profile_id"]
        print(f"Parsing profile {pid} ({i+1}/{len(raw)})...")
        experiences = parse_experience_with_llm(pid, row["country"], row["experience"])
        all_experiences.extend(experiences)
        print(f"  -> {len(experiences)} jobs extracted")

    print(f"\nDone. Total experience entries: {len(all_experiences)}")
    all_experiences[:5]
else:
    print(f"SKIPPED parsing. Using existing {PARSED_OUTPUT}")


## Export to csv

In [ ]:
from dataclasses import asdict

if RUN_PARSING:
    df_experiences = pd.DataFrame([asdict(e) for e in all_experiences])
    df_experiences.to_csv(PARSED_OUTPUT, index=False)
    print(f"Saved {len(df_experiences)} rows to {PARSED_OUTPUT}")
    df_experiences.head()
else:
    print(f"SKIPPED export. Using existing {PARSED_OUTPUT}")

## Categorize Job Titles
Classify each role title into one of three labels used in later steps.

- **Traditional Software Development**: Roles primarily focused on coding and software engineering.
- **Low-Code/No-Code Development**: Roles primarily building solutions with LCNC platforms.
- **Other**: Roles that do not clearly fit the two development categories.

Classification is based on the job title only; ambiguous titles are assigned to the most likely category.

In [ ]:
if RUN_CLASSIFICATION:
    df_experiences = pd.read_csv(PARSED_OUTPUT)
    print(f"Loaded {len(df_experiences)} rows from {PARSED_OUTPUT}")

    CLASSIFY_PROMPT = """
    You are an expert in software development role classification. Your task is to classify a given job role into exactly one of the following three categories.

    Categories

    1. Traditional Software Development Roles that primarily involve writing code using traditional programming languages and software engineering practices. This includes, but is not limited to: Software Engineer, Developer, Backend Developer, Frontend Developer, Full Stack Developer, Mobile Developer, DevOps Engineer, Data Scientist, Network Engineer, Embedded Systems Developer, Systems Programmer.

    2. Low-Code/No-Code (LCNC) Development Roles where the person primarily works with an LCNC platform to build applications with minimal or no traditional coding. This includes roles using platforms such as Power Platform (Power Apps, Power Automate, Power BI), Mendix, OutSystems, Appian, AppSheet, Webflow, Bubble, Framer, Zapier, Make (Integromat), and Airtable. Often found in a citizen developer, business analyst, or digital automation context.

    3. Other Roles that do not clearly fit into either category above. This includes management, sales, design, HR, and unrelated technical roles.

    Instructions Classify the given job role into exactly one of the three categories. Base your classification solely on the job title or role provided. If a role is ambiguous, choose the most likely category based on common industry usage. Return only the category name, nothing else.

    Output format Return only one of these exact strings: "Traditional Software Development", "Low-Code/No-Code Development", or "Other".

    Job title: {job_title}"""

    VALID_CATEGORIES = {
        "Traditional Software Development",
        "Low-Code/No-Code Development",
        "Other",
    }


    def classify_job(job_title: str) -> str:
        """Classify a single job title using GeminiClient."""
        prompt = CLASSIFY_PROMPT.format(job_title=job_title)

        for attempt in range(MAX_RETRIES):
            try:
                label = client.generate_text(prompt).strip()
                time.sleep(_RATE_LIMIT_DELAY)
                return label if label in VALID_CATEGORIES else "Other"

            except ClientError as e:
                if e.code == 429:
                    wait = 60
                    print(f"  Rate limited. Waiting {wait}s (attempt {attempt+1})...")
                    time.sleep(wait)
                else:
                    print(f"  API error for '{job_title}': {e}")
                    break
            except Exception as e:
                print(f"  Unexpected error for '{job_title}': {e}")
                break

        return "ERROR"


    # --- Classify all parsed experiences ---
    labels = []
    total = len(df_experiences)

    for i, row in df_experiences.iterrows():
        title = row["job_title"]

        if title == "PARSE_ERROR":
            labels.append("ERROR")
            continue

        print(f"Classifying [{i+1}/{total}] '{title}'...")
        label = classify_job(title)
        labels.append(label)

    df_classified = df_experiences.copy()
    df_classified["job_label"] = labels

    df_classified.to_csv(CLASSIFIED_OUTPUT, index=False)

    print(f"\nDone. Saved {len(df_classified)} classified entries to {CLASSIFIED_OUTPUT}")
    df_classified.head(10)
else:
    print(f"SKIPPED classification. Using existing {CLASSIFIED_OUTPUT}")


## Classify Seniority Level
Assign each job title a seniority score from **0** to **7** based on title cues.

- **0**: Not related to software, IT, tech, or business.
- **1-2**: Intern/student/trainee or junior/entry-level roles.
- **3**: Mid-level default when no seniority cue is present.
- **4-5**: Senior/staff/expert and lead/principal roles.
- **6-7**: Manager/director and executive/VP/C-suite roles.

If a title includes multiple cues, the higher level is used.

In [ ]:
if RUN_SENIORITY:
    df_classified = pd.read_csv(CLASSIFIED_OUTPUT)
    print(f"Loaded {len(df_classified)} rows from {CLASSIFIED_OUTPUT}")

    SENIORITY_PROMPT = """You are an expert in career-level classification for software development and IT professionals. Given a job title, assign a seniority level from 0 to 7 based ONLY on the title itself.

Context: These job titles come from LinkedIn profiles of software developers and IT professionals. Many profiles include early-career or side jobs unrelated to software/IT/tech or business in general.

Seniority scale:
0 = Not related to software development, IT, tech, or business (e.g. Waiter, Waitress, Warehouse Worker, Cashier, Barista, Retail Associate, Volunteer, Wedding Photographer)
1 = Intern / Student / Trainee (e.g. Software Engineer Internship, Working Student, Trainee, Tech Trainee)
2 = Junior / Entry-level (e.g. Junior Developer, Graduate Associate, Junior Systems Engineer)
3 = Mid-level — no seniority prefix (e.g. Software Engineer, Developer, Test Engineer, Web Developer, Analyst, Consultant)
4 = Senior / Staff / Expert (e.g. Senior Software Engineer, Staff Engineer, Operations Expert)
5 = Lead / Principal (e.g. Tech Lead, Team Lead, Principal Engineer, Chapter Lead)
6 = Manager / Director (e.g. Engineering Manager, IT Manager, Director of Operations, Program Manager)
7 = Executive / C-suite / VP (e.g. CEO, CTO, CIO, VP of Engineering, Managing Director)

Rules:
- If the job title contains NO seniority keywords AND is clearly unrelated to software development, IT, technology, or business (food service, retail, manual labor, volunteering, photography, sports coaching, etc.), return 0.
- If the title contains "intern", "internship", "trainee", "student", or "working student", return 1.
- If the title contains "junior" or "entry-level" or "graduate", return 2.
- If the title contains "senior", "staff", or "expert" (but not "lead" or "manager"), return 4.
- If the title contains "lead" or "principal", return 5.
- If the title contains "manager", "director", "head of", or "program manager", return 6.
- If the title contains "chief", "CEO", "CTO", "CIO", "CFO", "COO", "VP", "vice president", or "managing director", return 7.
- If none of the above apply, return 3 (mid-level default).
- When multiple indicators conflict (e.g. "Senior Manager"), choose the HIGHER level.

Return ONLY a single digit (0, 1, 2, 3, 4, 5, 6, or 7). Nothing else.

Job title: {job_title}"""

    VALID_SENIORITY = {"0", "1", "2", "3", "4", "5", "6", "7"}


    def classify_seniority(job_title: str) -> int:
        """Classify seniority level (0-7) for a single job title using GeminiClient."""
        prompt = SENIORITY_PROMPT.format(job_title=job_title)

        for attempt in range(MAX_RETRIES):
            try:
                result = client.generate_text(prompt).strip()
                if result in VALID_SENIORITY:
                    time.sleep(_RATE_LIMIT_DELAY)
                    return int(result)
                print(f"  Invalid response '{result}' for '{job_title}' (attempt {attempt+1})")

            except ClientError as e:
                if e.code == 429:
                    wait = 60
                    print(f"  Rate limited. Waiting {wait}s (attempt {attempt+1})...")
                    time.sleep(wait)
                else:
                    print(f"  API error for '{job_title}': {e}")
                    break
            except Exception as e:
                print(f"  Unexpected error for '{job_title}': {e}")
                break

        return -1  # error fallback


    # --- Classify seniority for all entries ---
    seniority_levels = []
    total = len(df_classified)

    for i, row in df_classified.iterrows():
        title = row["job_title"]

        if title == "PARSE_ERROR":
            seniority_levels.append(-1)
            continue

        print(f"Classifying seniority [{i+1}/{total}] '{title}'...")
        level = classify_seniority(title)
        seniority_levels.append(level)

    df_classified["seniority_level"] = seniority_levels

    df_classified.to_csv(SENIORITY_OUTPUT, index=False)

    print(f"\nDone. Saved {len(df_classified)} entries with seniority levels to {SENIORITY_OUTPUT}")
    print(f"\nSeniority distribution:")
    print(df_classified["seniority_level"].value_counts().sort_index())
    df_classified.head(10)
else:
    print(f"SKIPPED seniority classification. Using existing {SENIORITY_OUTPUT}")


## Duration
Compute how long each person stayed in each role, measured in months.

- Uses `start_date` and `end_date` from the parsed experience data.
- If `end_date` is missing (current role), today's date is used.
- If `start_date` is missing or invalid, duration is left empty.
- Negative durations are clipped to 0 months.

In [ ]:
if RUN_DURATION:
    from datetime import datetime

    df = pd.read_csv(SENIORITY_OUTPUT)
    print(f"Loaded {len(df)} rows from {SENIORITY_OUTPUT}")

    def parse_date(date_str):
        """Parse a date string in 'YYYY-MM' or 'YYYY' format to a datetime object."""
        if pd.isna(date_str) or str(date_str).strip() == "":
            return None
        s = str(date_str).strip()
        if len(s) == 7 and s[4] == "-":
            try:
                return datetime.strptime(s, "%Y-%m")
            except ValueError:
                return None
        if len(s) == 4 and s.isdigit():
            try:
                return datetime(int(s), 1, 1)
            except ValueError:
                return None
        return None

    def calc_duration_months(start_str, end_str):
        """Calculate duration in months between two date strings.
        Uses today's date if end_str is null (current role).
        Returns None if start_str is null.
        """
        start = parse_date(start_str)
        if start is None:
            return None
        end = parse_date(end_str)
        if end is None:
            end = datetime.today()
        months = (end.year - start.year) * 12 + (end.month - start.month)
        return max(months, 0)

    df["duration_months"] = df.apply(
        lambda row: calc_duration_months(row["start_date"], row["end_date"]), axis=1
    )
    df["duration_months"] = df["duration_months"].astype("Int64")

    df.to_csv(DURATION_OUTPUT, index=False)
    print(f"Done. Saved {len(df)} entries to {DURATION_OUTPUT}")
    print(f"\nDuration stats:")
    print(df["duration_months"].describe())
    df.head(10)
else:
    print(f"SKIPPED duration calculation. Using existing {DURATION_OUTPUT}")

## Profile Classification
Classify entire profiles based on roles they had in the past. A profile is classified as **Mixed** only when both LCNC and Traditional roles are roughly equally represented (ratio >= 40%). Otherwise the dominant type wins.

- **LCNC**: Only LCNC roles (no Traditional), or LCNC roles significantly outnumber Traditional roles (ratio < 40%).
- **Traditional Software Development**: Only Traditional roles (no LCNC), or Traditional roles significantly outnumber LCNC roles (ratio < 40%).
- **Mixed**: Both LCNC and Traditional roles present, with the minority type making up at least 40% of the majority type.
- **Other**: No LCNC and no Traditional Software Development roles.

In [ ]:
if RUN_PROFILES:
    df = pd.read_csv(DURATION_OUTPUT)
    print(f"Loaded {len(df)} rows from {DURATION_OUTPUT}")

    MIXED_THRESHOLD = 0.4

    def classify_profile(group):
        """Classify a profile based on the job_label distribution of all its roles.
        'Mixed' only applies when min(lcnc, trad) / max(lcnc, trad) >= MIXED_THRESHOLD.
        Otherwise the dominant type wins.
        """
        labels = group["job_label"].value_counts()
        lcnc_count = labels.get("Low-Code/No-Code Development", 0)
        trad_count = labels.get("Traditional Software Development", 0)

        if lcnc_count > 0 and trad_count > 0:
            ratio = min(lcnc_count, trad_count) / max(lcnc_count, trad_count)
            if ratio >= MIXED_THRESHOLD:
                return "Mixed"
            return "LCNC" if lcnc_count > trad_count else "Traditional Software Development"
        elif lcnc_count > 0:
            return "LCNC"
        elif trad_count > 0:
            return "Traditional Software Development"
        else:
            return "Other"

    profile_labels = (
        df.groupby("profile_id")
        .apply(classify_profile)
        .rename("profile_label")
    )

    df = df.merge(profile_labels, on="profile_id", how="left")

    df.to_csv(PROFILE_OUTPUT, index=False)
    print(f"Done. Saved {len(df)} entries to {PROFILE_OUTPUT}")
    print(f"\nProfile classification distribution:")
    print(df.drop_duplicates("profile_id")["profile_label"].value_counts())
    df.head(10)
else:
    print(f"SKIPPED profile classification. Using existing {PROFILE_OUTPUT}")